In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.profiler import profile, record_function, ProfilerActivity
import time

# ── device ──────────────────────────────────────────────────────────────────
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── data loading (same as your original) ────────────────────────────────────
mnisttrain = pd.read_csv("/kaggle/input/datasets/tomatopdf/mnist-mine/mnist_train.csv")
mnisttest  = pd.read_csv("/kaggle/input/datasets/tomatopdf/mnist-mine/mnist_test.csv")

X_train = mnisttrain.iloc[:20000, 1:].values.astype(np.float32)
Y_train = mnisttrain.iloc[:20000, 0].values

X_test  = mnisttest.iloc[-2000:, 1:].values.astype(np.float32)
Y_test  = mnisttest.iloc[-2000:, 0].values

# same normalization
X_train = (X_train / 127.5) - 1
X_test  = (X_test  / 127.5) - 1

# NOTE: no bias trick needed — nn.Linear handles bias internally
# convert to tensors
X_train = torch.tensor(X_train, device=device)
Y_train = torch.tensor(Y_train, dtype=torch.long, device=device)
X_test  = torch.tensor(X_test,  device=device)
Y_test  = torch.tensor(Y_test,  dtype=torch.long, device=device)

# train/val split
val_size   = int(0.1 * len(X_train))
X_val      = X_train[-val_size:]
Y_val      = Y_train[-val_size:]
X_train    = X_train[:-val_size]
Y_train    = Y_train[:-val_size]

# ── model — exact same architecture ─────────────────────────────────────────
class MnistANN(nn.Module):
    def __init__(self, n_h=64, drop_prob=0.1):
        super().__init__()
        # your w1, w2, w3 — bias=True replicates your bias trick
        self.fc1 = nn.Linear(784, n_h, bias=True)
        self.fc2 = nn.Linear(n_h,  n_h, bias=True)
        self.fc3 = nn.Linear(n_h,  10,  bias=True)
        self.drop = nn.Dropout(drop_prob)

        # same initialization as your weight_initialization(uniform=False)
        # your code: normal(0, 1/sqrt(input_units))
        nn.init.normal_(self.fc1.weight, 0, 1/np.sqrt(784))
        nn.init.normal_(self.fc2.weight, 0, 1/np.sqrt(n_h))
        nn.init.normal_(self.fc3.weight, 0, 1/np.sqrt(n_h))

    def forward(self, x):
        x = self.drop(torch.tanh(self.fc1(x)))
        x = self.drop(torch.tanh(self.fc2(x)))
        x = self.fc3(x)          # raw logits — CrossEntropyLoss expects this
        return x

model = MnistANN(n_h=64).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

# ── training loop ────────────────────────────────────────────────────────────
batch_size  = 128
max_epochs  = 20
n_train     = X_train.shape[0]
num_batches = n_train // batch_size

for epoch in range(max_epochs):
    model.train()
    for i in range(num_batches):
        xb = X_train[i*batch_size : (i+1)*batch_size]
        yb = Y_train[i*batch_size : (i+1)*batch_size]

        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss   = criterion(val_logits, Y_val).item()
        val_acc    = (val_logits.argmax(1) == Y_val).float().mean().item()
    print(f"Epoch {epoch+1:2d} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

Using device: cuda:0
GPU: Tesla T4
Epoch  1 | val_loss=1.1272 | val_acc=0.6475
Epoch  2 | val_loss=0.8923 | val_acc=0.6915
Epoch  3 | val_loss=0.5926 | val_acc=0.8200
Epoch  4 | val_loss=0.5890 | val_acc=0.8360
Epoch  5 | val_loss=0.4643 | val_acc=0.8640
Epoch  6 | val_loss=0.3509 | val_acc=0.8915
Epoch  7 | val_loss=0.3830 | val_acc=0.8865
Epoch  8 | val_loss=0.3065 | val_acc=0.9095
Epoch  9 | val_loss=0.3163 | val_acc=0.9115
Epoch 10 | val_loss=0.3254 | val_acc=0.9080
Epoch 11 | val_loss=0.3050 | val_acc=0.9090
Epoch 12 | val_loss=0.4278 | val_acc=0.8745
Epoch 13 | val_loss=0.2808 | val_acc=0.9215
Epoch 14 | val_loss=0.3281 | val_acc=0.9055
Epoch 15 | val_loss=0.3303 | val_acc=0.9050
Epoch 16 | val_loss=0.2705 | val_acc=0.9185
Epoch 17 | val_loss=0.3074 | val_acc=0.9040
Epoch 18 | val_loss=0.2593 | val_acc=0.9230
Epoch 19 | val_loss=0.2409 | val_acc=0.9295
Epoch 20 | val_loss=0.2579 | val_acc=0.9245


In [2]:
model.train()
xb = X_train[:batch_size]
yb = Y_train[:batch_size]

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    for _ in range(20):  # repeat so averages are stable
        with record_function("forward"):
            logits = model(xb)
            loss   = criterion(logits, yb)
        with record_function("backward"):
            optimizer.zero_grad()
            loss.backward()
        with record_function("optimizer_step"):
            optimizer.step()
        torch.cuda.synchronize()

print(prof.key_averages().table(sort_by="cuda_time_total"))

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                forward         0.00%       0.000us         0.00%       0.000us       0.000us      19.250ms       384.83%      19.250ms     962.495us           0 B           0 B           0 B           0 

In [3]:
def benchmark_batch(batch_size, n_runs=200):
    model.train()
    # build a batch of this size (repeat if needed)
    reps   = (batch_size // X_train.shape[0]) + 1
    X_big  = X_train.repeat(reps, 1)[:batch_size]
    Y_big  = Y_train.repeat(reps)[:batch_size]

    # warmup
    for _ in range(10):
        logits = model(X_big)
        loss   = criterion(logits, Y_big)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    torch.cuda.synchronize()

    t0 = time.time()
    for _ in range(n_runs):
        logits = model(X_big)
        loss   = criterion(logits, Y_big)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    torch.cuda.synchronize()
    elapsed = (time.time() - t0) / n_runs * 1000  # ms per step

    # FLOPS estimate: 2*M*K*N per matmul, three layers
    # forward: (B,784)x(784,64) + (B,64)x(64,64) + (B,64)x(10,64).T
    flops_forward = 2 * batch_size * (784*64 + 64*64 + 64*10)
    flops_total   = flops_forward * 3  # rough: backward ~2x forward
    tflops        = flops_total / elapsed / 1e9

    print(f"batch={batch_size:5d} | {elapsed:.3f} ms/step | "
          f"~{tflops:.4f} TFLOPS | "
          f"T4 peak FP32=8.1 TFLOPS → utilization ~{tflops/8.1*100:.3f}%")

for bs in [32, 128, 512, 1024, 4096, 8192, 16384]:
    benchmark_batch(bs)

batch=   32 | 1.261 ms/step | ~0.0084 TFLOPS | T4 peak FP32=8.1 TFLOPS → utilization ~0.103%
batch=  128 | 1.340 ms/step | ~0.0315 TFLOPS | T4 peak FP32=8.1 TFLOPS → utilization ~0.389%
batch=  512 | 1.358 ms/step | ~0.1242 TFLOPS | T4 peak FP32=8.1 TFLOPS → utilization ~1.534%
batch= 1024 | 1.426 ms/step | ~0.2365 TFLOPS | T4 peak FP32=8.1 TFLOPS → utilization ~2.920%
batch= 4096 | 1.420 ms/step | ~0.9503 TFLOPS | T4 peak FP32=8.1 TFLOPS → utilization ~11.732%
batch= 8192 | 1.362 ms/step | ~1.9816 TFLOPS | T4 peak FP32=8.1 TFLOPS → utilization ~24.464%
batch=16384 | 1.840 ms/step | ~2.9338 TFLOPS | T4 peak FP32=8.1 TFLOPS → utilization ~36.220%


In [4]:
def benchmark_width(n_h, batch_size=4096, n_runs=100):
    wide_model = MnistANN(n_h=n_h).to(device)
    wide_opt   = torch.optim.SGD(wide_model.parameters(), lr=0.5)
    wide_model.train()

    reps  = (batch_size // X_train.shape[0]) + 1
    X_big = X_train.repeat(reps, 1)[:batch_size]
    Y_big = Y_train.repeat(reps)[:batch_size]

    for _ in range(10):  # warmup
        logits = wide_model(X_big)
        loss   = criterion(logits, Y_big)
        wide_opt.zero_grad(); loss.backward(); wide_opt.step()
    torch.cuda.synchronize()

    t0 = time.time()
    for _ in range(n_runs):
        logits = wide_model(X_big)
        loss   = criterion(logits, Y_big)
        wide_opt.zero_grad(); loss.backward(); wide_opt.step()
    torch.cuda.synchronize()
    elapsed = (time.time() - t0) / n_runs * 1000

    flops   = 2 * batch_size * (784*n_h + n_h*n_h + n_h*10) * 3
    tflops  = flops / elapsed / 1e9
    params  = sum(p.numel() for p in wide_model.parameters())
    print(f"n_h={n_h:5d} | params={params:8d} | {elapsed:.2f} ms | "
          f"{tflops:.3f} TFLOPS | ~{tflops/8.1*100:.1f}% of T4 peak")

for n_h in [64, 256, 1024, 2048, 4096]:
    benchmark_width(n_h)

n_h=   64 | params=   55050 | 1.29 ms | 1.049 TFLOPS | ~12.9% of T4 peak
n_h=  256 | params=  269322 | 1.74 ms | 3.796 TFLOPS | ~46.9% of T4 peak
n_h= 1024 | params= 1863690 | 10.24 ms | 4.469 TFLOPS | ~55.2% of T4 peak
n_h= 2048 | params= 5824522 | 30.31 ms | 4.719 TFLOPS | ~58.3% of T4 peak
n_h= 4096 | params=20037642 | 107.96 ms | 4.560 TFLOPS | ~56.3% of T4 peak


In [5]:
# Check if Tensor Cores are being invoked
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

scaler = torch.amp.GradScaler('cuda')
wide_model = MnistANN(n_h=2048).to(device)
wide_opt   = torch.optim.SGD(wide_model.parameters(), lr=0.5)

reps  = (4096 // X_train.shape[0]) + 1
X_big = X_train.repeat(reps, 1)[:4096]
Y_big = Y_train.repeat(reps)[:4096]

# FP32 baseline
wide_model.train()
for _ in range(10): wide_model(X_big)  # warmup
torch.cuda.synchronize()
t0 = time.time()
for _ in range(200):
    logits = wide_model(X_big)
    loss   = criterion(logits, Y_big)
    wide_opt.zero_grad(); loss.backward(); wide_opt.step()
torch.cuda.synchronize()
t_fp32 = (time.time()-t0)/200*1000

# FP16 with autocast
for _ in range(10):
    with torch.amp.autocast('cuda'): wide_model(X_big)
torch.cuda.synchronize()
t0 = time.time()
for _ in range(200):
    with torch.amp.autocast('cuda'):
        logits = wide_model(X_big)
        loss   = criterion(logits, Y_big)
    wide_opt.zero_grad()
    scaler.scale(loss).backward()
    scaler.step(wide_opt)
    scaler.update()
torch.cuda.synchronize()
t_fp16 = (time.time()-t0)/200*1000

print(f"FP32: {t_fp32:.2f} ms/step")
print(f"FP16: {t_fp16:.2f} ms/step")
print(f"Speedup: {t_fp32/t_fp16:.2f}x")
print()
print("On T4: FP16 activates Tensor Cores (65 TFLOPS) vs FP32 CUDA cores (8.1 TFLOPS)")
print("Theoretical max speedup: ~8x for compute-bound ops")
print("Real speedup will be less due to memory bandwidth and kernel launch overhead")

# Then in your FP16 experiment, after running it:
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
) as prof:
    for _ in range(20):
        with torch.amp.autocast('cuda'):
            logits = wide_model(X_big)
            loss = criterion(logits, Y_big)
        scaler.scale(loss).backward()
        scaler.step(wide_opt)
        scaler.update()
    torch.cuda.synchronize()

# Look specifically for these kernel name patterns in the output:
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))
# Tensor Core kernels contain "h884" or "hmma" or "s16816gemm" in their names
# Regular CUDA core kernels contain "sgemm" (single precision gemm)

FP32: 31.22 ms/step
FP16: 7.34 ms/step
Speedup: 4.25x

On T4: FP16 activates Tensor Cores (65 TFLOPS) vs FP32 CUDA cores (8.1 TFLOPS)
Theoretical max speedup: ~8x for compute-bound ops
Real speedup will be less due to memory bandwidth and kernel launch overhead
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::linear         0.44%     849.652us         8.00%      15.373ms     128.107us       0.000us    

In [6]:
import torch
import time

device = 'cuda:0'

def roofline_point(M, K, N, dtype, n_runs=100):
    A = torch.randn(M, K, device=device, dtype=dtype)
    B = torch.randn(K, N, device=device, dtype=dtype)

    for _ in range(10):
        C = torch.mm(A, B)
    torch.cuda.synchronize()

    t0 = time.time()
    for _ in range(n_runs):
        C = torch.mm(A, B)
    torch.cuda.synchronize()
    elapsed = (time.time() - t0) / n_runs * 1000

    flops = 2 * M * K * N
    bytes_moved = (M*K + K*N + M*N) * (2 if dtype == torch.float16 else 4)
    intensity = flops / bytes_moved
    peak = 65.0 if dtype == torch.float16 else 8.1
    tflops = flops / elapsed / 1e9
    label = "FP16" if dtype == torch.float16 else "FP32"

    print(f"{label} [{M:5d}x{K:5d}x{N:5d}] | elapsed={elapsed:.3f}ms | "
          f"{tflops:.2f} TFLOPS | {tflops/peak*100:.1f}% peak | "
          f"intensity={intensity:.1f} FLOPS/byte")

print("--- Small (your network's actual matrices) ---")
roofline_point(128,  784, 64,   torch.float32)
roofline_point(128,  784, 64,   torch.float16)

print("\n--- Medium ---")
roofline_point(1024, 1024, 1024, torch.float32)
roofline_point(1024, 1024, 1024, torch.float16)

print("\n--- Large (where GPUs are designed to operate) ---")
roofline_point(4096, 4096, 4096, torch.float32)
roofline_point(4096, 4096, 4096, torch.float16)

print("\n--- Square matrices matching LLM projection sizes ---")
roofline_point(2048, 768,  3072, torch.float16)  # GPT-2 FFN layer
roofline_point(2048, 4096, 4096, torch.float16)  # LLaMA-7B attention


--- Small (your network's actual matrices) ---
FP32 [  128x  784x   64] | elapsed=0.024ms | 0.53 TFLOPS | 6.5% peak | intensity=20.2 FLOPS/byte
FP16 [  128x  784x   64] | elapsed=0.025ms | 0.52 TFLOPS | 0.8% peak | intensity=40.5 FLOPS/byte

--- Medium ---
FP32 [ 1024x 1024x 1024] | elapsed=0.383ms | 5.61 TFLOPS | 69.3% peak | intensity=170.7 FLOPS/byte
FP16 [ 1024x 1024x 1024] | elapsed=0.105ms | 20.46 TFLOPS | 31.5% peak | intensity=341.3 FLOPS/byte

--- Large (where GPUs are designed to operate) ---
FP32 [ 4096x 4096x 4096] | elapsed=33.514ms | 4.10 TFLOPS | 50.6% peak | intensity=682.7 FLOPS/byte
FP16 [ 4096x 4096x 4096] | elapsed=6.003ms | 22.90 TFLOPS | 35.2% peak | intensity=1365.3 FLOPS/byte

--- Square matrices matching LLM projection sizes ---
FP16 [ 2048x  768x 3072] | elapsed=0.461ms | 20.97 TFLOPS | 32.3% peak | intensity=472.6 FLOPS/byte
FP16 [ 2048x 4096x 4096] | elapsed=2.826ms | 24.32 TFLOPS | 37.4% peak | intensity=1024.0 FLOPS/byte


In [7]:
# Fix compute, vary memory pressure
# If compute is the bottleneck, halving data size shouldn't help much
# If memory is the bottleneck, halving data size should give ~2x speedup

import torch, time

device = 'cuda:0'

def time_mm(M, K, N, dtype, n_runs=200):
    A = torch.randn(M, K, device=device, dtype=dtype)
    B = torch.randn(K, N, device=device, dtype=dtype)
    for _ in range(10): torch.mm(A, B)
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_runs): torch.mm(A, B)
    torch.cuda.synchronize()
    return (time.time() - t0) / n_runs * 1000

M = K = N = 4096

t32 = time_mm(M, K, N, torch.float32)
t16 = time_mm(M, K, N, torch.float16)

flops = 2 * M * K * N

bytes32 = M*K*4 + K*N*4 + M*N*4
bytes16 = M*K*2 + K*N*2 + M*N*2

tflops32 = flops / t32 / 1e9
tflops16 = flops / t16 / 1e9

bw32 = bytes32 / t32 / 1e6   # GB/s actually consumed
bw16 = bytes16 / t16 / 1e6

print(f"FP32: {tflops32:.2f} TFLOPS | memory consumed: {bw32:.1f} GB/s of 300 GB/s")
print(f"FP16: {tflops16:.2f} TFLOPS | memory consumed: {bw16:.1f} GB/s of 300 GB/s")
print(f"Speedup: {t32/t16:.2f}x")
print(f"Memory bytes ratio: {bytes32/bytes16:.1f}x  (FP32 moves 2x more data)")
print(f"Speedup / memory ratio: {(t32/t16)/(bytes32/bytes16):.2f}  (1.0 = purely memory bound)")

FP32: 4.02 TFLOPS | memory consumed: 5.9 GB/s of 300 GB/s
FP16: 22.88 TFLOPS | memory consumed: 16.8 GB/s of 300 GB/s
Speedup: 5.69x
Memory bytes ratio: 2.0x  (FP32 moves 2x more data)
Speedup / memory ratio: 2.84  (1.0 = purely memory bound)


In [8]:
import torch
import time

device = 'cuda:0'

# T4 specs
TOTAL_CUDA_CORES = 2560
BOOST_CLOCK_GHZ  = 1.590        # T4 boost clock
FLOPS_PER_CORE   = BOOST_CLOCK_GHZ * 2   # FMA = 2 FLOPS/cycle
TENSOR_CORES     = 320          # T4 has 320 Tensor Cores (2nd gen)

def measure(M, K, N, dtype, n_runs=500):
    A = torch.randn(M, K, device=device).to(dtype)
    B = torch.randn(K, N, device=device).to(dtype)

    # INT8 needs different path
    if dtype == torch.int8:
        fn = lambda: torch._int_mm(A, B)
    else:
        fn = lambda: torch.mm(A, B)

    for _ in range(20): fn()
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_runs): fn()
    torch.cuda.synchronize()
    elapsed = (time.time() - t0) / n_runs * 1000  # ms

    flops      = 2 * M * K * N
    tflops     = flops / elapsed / 1e9

    if dtype == torch.float32:
        peak            = 8.1
        cores_available = TOTAL_CUDA_CORES
        flops_per_core  = FLOPS_PER_CORE        # CUDA cores do FP32 FMA
        core_label      = "CUDA cores"
    elif dtype == torch.float16:
        peak            = 65.0
        # Tensor Cores do 16x16x16 = 4096 FLOPS per core per clock
        # but reported as TFLOPS across all Tensor Cores
        cores_available = TENSOR_CORES
        flops_per_core  = 4096 * BOOST_CLOCK_GHZ  # GFLOPS per Tensor Core
        core_label      = "Tensor Cores"
    else:  # int8
        peak            = 130.0
        cores_available = TENSOR_CORES
        flops_per_core  = 4096 * 2 * BOOST_CLOCK_GHZ  # INT8 = 2x FP16 throughput
        core_label      = "Tensor Cores (INT8)"

    utilization_pct    = tflops / peak * 100
    # effective cores = total throughput / throughput per core
    effective_cores    = (tflops * 1000) / flops_per_core   # both in GFLOPS
    cores_pct          = effective_cores / cores_available * 100

    print(f"\n{str(dtype).split('.')[-1].upper()} [{M}x{K}x{N}]")
    print(f"  Time:              {elapsed:.3f} ms")
    print(f"  Achieved:          {tflops:.2f} TFLOPS")
    print(f"  Peak utilization:  {utilization_pct:.1f}%")
    print(f"  Effective {core_label}: {effective_cores:.0f} / {cores_available} ({cores_pct:.1f}%)")

M = K = N = 4096

measure(M, K, N, torch.float32)
measure(M, K, N, torch.float16)
measure(M, K, N, torch.int8)


FLOAT32 [4096x4096x4096]
  Time:              35.179 ms
  Achieved:          3.91 TFLOPS
  Peak utilization:  48.2%
  Effective CUDA cores: 1229 / 2560 (48.0%)

FLOAT16 [4096x4096x4096]
  Time:              6.385 ms
  Achieved:          21.53 TFLOPS
  Peak utilization:  33.1%
  Effective Tensor Cores: 3 / 320 (1.0%)

INT8 [4096x4096x4096]
  Time:              8.241 ms
  Achieved:          16.68 TFLOPS
  Peak utilization:  12.8%
  Effective Tensor Cores (INT8): 1 / 320 (0.4%)


# calculating latency for different matrix sizes

In [9]:
import torch
import time
from torch.profiler import profile, ProfilerActivity

device = 'cuda:0'

# T4 specs
T4_FP32_PEAK  = 8.1    # TFLOPS (CUDA cores)
T4_FP16_PEAK  = 65.0   # TFLOPS (Tensor cores)
T4_BANDWIDTH  = 300.0  # GB/s

def get_kernel_type(M, K, N, dtype, n_profile_runs=5):
    """Run profiler and extract the actual kernel name."""
    if dtype == torch.int8:
        A = torch.randint(-128, 127, (M, K), device=device, dtype=torch.int8)
        B = torch.randint(-128, 127, (K, N), device=device, dtype=torch.int8)
        fn = lambda: torch._int_mm(A, B)
    else:
        A = torch.randn(M, K, device=device, dtype=dtype)
        B = torch.randn(K, N, device=device, dtype=dtype)
        fn = lambda: torch.mm(A, B)

    with profile(activities=[ProfilerActivity.CUDA]) as prof:
        for _ in range(n_profile_runs):
            fn()
        torch.cuda.synchronize()

    # extract all event keys — filter to CUDA kernels by checking cuda time
    kernel_names = []
    for e in prof.key_averages():
        try:
            # newer PyTorch versions use self_device_time_total
            cuda_time = e.self_device_time_total
        except AttributeError:
            try:
                cuda_time = e.self_cuda_time_total
            except AttributeError:
                cuda_time = 0

        if cuda_time > 0:
            kernel_names.append(e.key)

    # classify based on kernel name tokens
    for name in kernel_names:
        name_lower = name.lower()
        if any(tc in name_lower for tc in ['s1688', 'hmma', 'h884', 'tensorop', 'imma', 'i8816']):
            return "TENSOR CORE", name
        if any(cc in name_lower for cc in ['sgemm', 'hgemm_', 'igemm']):
            return "CUDA CORE", name

    return "UNKNOWN", kernel_names[0] if kernel_names else "none"


def benchmark(M, K, N, dtype, n_runs=500):
    """Measure latency and throughput for a single matmul."""
    if dtype == torch.int8:
        A = torch.randint(-128, 127, (M, K), device=device, dtype=torch.int8)
        B = torch.randint(-128, 127, (K, N), device=device, dtype=torch.int8)
        fn = lambda: torch._int_mm(A, B)
    else:
        A = torch.randn(M, K, device=device, dtype=dtype)
        B = torch.randn(K, N, device=device, dtype=dtype)
        fn = lambda: torch.mm(A, B)

    # warmup — essential, first runs include JIT/cuBLAS heuristic lookup cost
    for _ in range(20):
        fn()
    torch.cuda.synchronize()

    # timed runs
    t0 = time.time()
    for _ in range(n_runs):
        fn()
    torch.cuda.synchronize()
    elapsed_ms = (time.time() - t0) / n_runs * 1000

    # derived metrics
    flops        = 2 * M * K * N
    bytes_moved  = (M*K + K*N + M*N) * dtype_bytes(dtype)
    intensity    = flops / bytes_moved           # FLOPS/byte
    tflops       = flops / elapsed_ms / 1e9
    peak         = dtype_peak(dtype)
    ridge        = peak * 1e3 / T4_BANDWIDTH     # TFLOPS→GFLOPS, /GB/s = FLOPS/byte
    utilization  = tflops / peak * 100
    regime       = "COMPUTE" if intensity > ridge else "MEMORY"

    return {
        "elapsed_ms":  elapsed_ms,
        "tflops":      tflops,
        "utilization": utilization,
        "intensity":   intensity,
        "ridge":       ridge,
        "regime":      regime,
        "flops":       flops,
        "bytes":       bytes_moved,
    }

def dtype_bytes(dtype):
    return {torch.float32: 4, torch.float16: 2, torch.int8: 1}[dtype]

def dtype_peak(dtype):
    return {torch.float32: T4_FP32_PEAK, torch.float16: T4_FP16_PEAK, torch.int8: 130.0}[dtype]

def dtype_label(dtype):
    return {torch.float32: "FP32", torch.float16: "FP16", torch.int8: "INT8"}[dtype]

# ── matrix sizes to test ───────────────────────────────────────────────────
configs = [
    # (M,    K,    N,    label)
    (   4,    4,    4,  "miniscule"),
    (  64,   64,   64,  "tiny"),
    ( 128,  784,   64,  "MNIST"),
    ( 256,  256,  256,  "small"),
    ( 512,  512,  512,  "small-medium"),
    (1024, 1024, 1024,  "medium"),
    (2048,  768, 3072,  "GPT-2 FFN"),
    (2048, 4096, 4096,  "LLaMA attention"),
    (4096, 4096, 4096,  "large"),
]

dtypes = [torch.float32, torch.float16]

# ── run ────────────────────────────────────────────────────────────────────
print("=" * 100)
print(f"{'Label':<22} {'dtype':<6} {'M×K×N':<22} {'Latency':>9} {'TFLOPS':>8} "
      f"{'Util%':>7} {'Intensity':>11} {'Regime':>9} {'Hardware':<14}")
print("=" * 100)

for M, K, N, label in configs:
    for dtype in dtypes:
        stats        = benchmark(M, K, N, dtype)
        hw, kname    = get_kernel_type(M, K, N, dtype)
        dl           = dtype_label(dtype)
        shape_str    = f"{M}×{K}×{N}"

        # flag anomalies
        flags = []
        if stats["regime"] == "MEMORY" and stats["utilization"] < 5:
            flags.append("! latency-bound")
        if dtype == torch.float16 and hw == "CUDA CORE":
            flags.append("⚠ TC not used")

        flag_str = "  " + ", ".join(flags) if flags else ""

        print(f"{label:<22} {dl:<6} {shape_str:<22} "
              f"{stats['elapsed_ms']:>8.3f}ms "
              f"{stats['tflops']:>8.2f} "
              f"{stats['utilization']:>6.1f}% "
              f"{stats['intensity']:>10.1f} "
              f"{stats['regime']:>9} "
              f"{hw:<14}"
              f"{flag_str}")

    print("-" * 100)

print()
print("Ridge points (intensity above this = compute bound):")
print(f"  FP32: {T4_FP32_PEAK*1000/T4_BANDWIDTH:.1f} FLOPS/byte")
print(f"  FP16: {T4_FP16_PEAK*1000/T4_BANDWIDTH:.1f} FLOPS/byte")
print(f"  INT8: {130.0*1000/T4_BANDWIDTH:.1f} FLOPS/byte")
print()
print("Kernel classification key:")
print("  TENSOR CORE → kernel name contains s1688/hmma/h884/tensorop/imma")
print("  CUDA CORE   → kernel name contains sgemm/hgemm/igemm")

Label                  dtype  M×K×N                    Latency   TFLOPS   Util%   Intensity    Regime Hardware      
miniscule              FP32   4×4×4                     0.018ms     0.00    0.0%        0.7    MEMORY CUDA CORE       ! latency-bound
miniscule              FP16   4×4×4                     0.018ms     0.00    0.0%        1.3    MEMORY UNKNOWN         ! latency-bound
----------------------------------------------------------------------------------------------------
tiny                   FP32   64×64×64                  0.018ms     0.03    0.4%       10.7    MEMORY UNKNOWN         ! latency-bound
tiny                   FP16   64×64×64                  0.019ms     0.03    0.0%       21.3    MEMORY UNKNOWN         ! latency-bound
----------------------------------------------------------------------------------------------------
MNIST                  FP32   128×784×64                0.022ms     0.58    7.1%       20.2    MEMORY CUDA CORE     
MNIST                  FP16 

In [10]:
import torch
import time
from torch.profiler import profile, ProfilerActivity

device = 'cuda:0'

def dtype_bytes(dtype):
    return {torch.float32: 4, torch.float16: 2, torch.int8: 1}[dtype]

def dtype_label(dtype):
    return {torch.float32: "FP32", torch.float16: "FP16", torch.int8: "INT8"}[dtype]

configs = [
    (   4,    4,    4,  "miniscule"),
    (  64,   64,   64,  "tiny"),
    ( 128,  784,   64,  "MNIST layer"),
    ( 256,  256,  256,  "small"),
    ( 512,  512,  512,  "small-medium"),
    (1024, 1024, 1024,  "medium"),
    (2048,  768, 3072,  "GPT-2 FFN"),
    (2048, 4096, 4096,  "LLaMA attention"),
    (4096, 4096, 4096,  "large"),
]

dtypes = [torch.float32, torch.float16]

for M, K, N, label in configs:
    for dtype in dtypes:
        dl = dtype_label(dtype)

        if dtype == torch.int8:
            A = torch.randint(-128, 127, (M, K), device=device, dtype=torch.int8)
            B = torch.randint(-128, 127, (K, N), device=device, dtype=torch.int8)
            fn = lambda: torch._int_mm(A, B)
        else:
            A = torch.randn(M, K, device=device, dtype=dtype)
            B = torch.randn(K, N, device=device, dtype=dtype)
            fn = lambda: torch.mm(A, B)

        # warmup outside profiler
        for _ in range(20):
            fn()
        torch.cuda.synchronize()

        with profile(
            activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
            record_shapes=True,
        ) as prof:
            for _ in range(50):
                fn()
            torch.cuda.synchronize()

        print("=" * 120)
        print(f"  {dl}  [{M}×{K}×{N}]  ({label})")
        print("=" * 120)
        print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

  FP32  [4×4×4]  (miniscule)
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm        33.40%     710.276us        99.54%       2.117ms      42.336us     177.658us       100.00%     177.658us       3.553us            50  
void gemmSN_NN_kernel<float, 256, 4, 2, 8, 4, 4, fal...         0.00%       0.000us         0.00%       0.000us       0.000us     177.658us       100.00%     177.658us       4.13

# list of cuBLAS kernels